[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/05_attention.ipynb)

# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch
import math

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def scaled_dot_product_attention(Q, K, V):
  # Find the column or last dim of K
  d_k = K.size(-1)

  # Attention scores
  atten_scores = Q @ K.transpose(1,2) # [b, seq_q, d_k] [b, d_k, seq_k] >> [b, seq_q, seq_k]
  atten_scores = atten_scores / math.sqrt(d_k)

  # Find the attention weights
  atten_weights = torch.softmax(atten_scores, dim=-1)

  # context vector
  context_vec = atten_weights @ V # [b, seq_q, seq_k] x [b, seq_q, d_v] >> [b, seq_q, d_v]

  return context_vec


In [13]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)

Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


In [14]:
# ✅ SUBMIT
from torch_judge import check
check("attention")


🧪 Testing: Softmax Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.7ms)
  ✅ [2/4] Numerical correctness (7.9ms)
  ✅ [3/4] Gradient check (27.5ms)
  ✅ [4/4] Cross-attention (seq_q != seq_k) (0.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.7ms total)
  Progress saved. Run status() to see your dashboard.

